# Notebook 5 — Feature Engineering

## Load train, validation, and test sets

In [1]:
import pandas as pd

train = pd.read_csv("../artifacts/03_train.csv")
val = pd.read_csv("../artifacts/03_val.csv")
test = pd.read_csv("../artifacts/03_test.csv")

print("train:", train.shape)
print("val:  ", val.shape)
print("test: ", test.shape)

train: (67529, 22)
val:   (14470, 22)
test:  (14471, 22)


**Note:** Loaded all three splits with the same shapes saved in Notebook 3 — 
67,529 train / 14,470 val / 14,471 test rows, 22 columns each.

## Select features based on Notebook 4 findings

In [2]:
numeric_features = [
    "n_items", "total_price", "total_freight", "avg_price",
    "n_unique_sellers", "n_unique_products",
    "total_payment_value", "n_payment_installments", "n_payment_methods"
]

categorical_features = ["customer_state"]

target = "is_late"

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['n_items', 'total_price', 'total_freight', 'avg_price', 'n_unique_sellers', 'n_unique_products', 'total_payment_value', 'n_payment_installments', 'n_payment_methods']
Categorical features: ['customer_state']


**Note:** Feature lists confirmed — 9 numeric features (order/payment 
aggregates) and 1 categorical feature (customer_state), matching the 
decisions from Notebook 4's EDA summary.

## Add purchase_month feature (captures seasonality)

In [3]:
for df in [train, val, test]:
    df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month

categorical_features.append("purchase_month")

train[["order_purchase_timestamp", "purchase_month"]].head()

,order_purchase_timestamp,purchase_month
0,2016-09-15 12:16:38,9
1,2016-10-03 09:44:50,10
2,2016-10-03 16:56:50,10
3,2016-10-03 21:13:36,10
4,2016-10-03 22:06:03,10


**Note:** purchase_month extracted correctly (e.g., September → 9, 
October → 10). This will be one-hot encoded like customer_state, so the 
model can learn seasonal patterns (like the Feb-Mar spike found in EDA) 
without assuming a false numeric order between months.

## Handle missing values (fit imputer on train only)

In [4]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

train[numeric_features] = imputer.fit_transform(train[numeric_features])
val[numeric_features] = imputer.transform(val[numeric_features])
test[numeric_features] = imputer.transform(test[numeric_features])

print("Missing values after imputation:")
print("train:", train[numeric_features].isna().sum().sum())
print("val:  ", val[numeric_features].isna().sum().sum())
print("test: ", test[numeric_features].isna().sum().sum())

Missing values after imputation:
train: 0
val:   0
test:  0


**Note:** All missing values in numeric features successfully imputed 
using the median (calculated from train only, then applied to val/test). 
Zero missing values remain in any split.

## One-hot encode categorical features (fit on train only)

In [5]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_cat_encoded = encoder.fit_transform(train[categorical_features])
val_cat_encoded = encoder.transform(val[categorical_features])
test_cat_encoded = encoder.transform(test[categorical_features])

encoded_cols = encoder.get_feature_names_out(categorical_features)

print("Number of encoded columns:", len(encoded_cols))
print(encoded_cols[:10])

Number of encoded columns: 39
['customer_state_AC' 'customer_state_AL' 'customer_state_AM'
 'customer_state_AP' 'customer_state_BA' 'customer_state_CE'
 'customer_state_DF' 'customer_state_ES' 'customer_state_GO'
 'customer_state_MA']


**Note:** One-hot encoding produced 39 columns as expected: 27 for 
customer_state + 12 for purchase_month. Fitted on train only, then 
applied to val/test using the same encoding scheme.

## Combine numeric and encoded categorical features into final tables

In [6]:
train_encoded_df = pd.DataFrame(train_cat_encoded, columns=encoded_cols, index=train.index)
val_encoded_df = pd.DataFrame(val_cat_encoded, columns=encoded_cols, index=val.index)
test_encoded_df = pd.DataFrame(test_cat_encoded, columns=encoded_cols, index=test.index)

train_final = pd.concat([train[numeric_features + [target]], train_encoded_df], axis=1)
val_final = pd.concat([val[numeric_features + [target]], val_encoded_df], axis=1)
test_final = pd.concat([test[numeric_features + [target]], test_encoded_df], axis=1)

print("train_final:", train_final.shape)
print("val_final:  ", val_final.shape)
print("test_final: ", test_final.shape)

train_final: (67529, 49)
val_final:   (14470, 49)
test_final:  (14471, 49)


**Note:** Final feature tables built successfully — 49 columns each 
(9 numeric + 1 target + 39 one-hot encoded), with row counts matching 
the original splits exactly.

## Save feature tables, fitted transformers, and feature list

In [7]:
import joblib

# Save the final feature tables
train_final.to_csv("../artifacts/05_train_features.csv", index=False)
val_final.to_csv("../artifacts/05_val_features.csv", index=False)
test_final.to_csv("../artifacts/05_test_features.csv", index=False)

# Save the fitted transformers (so production can reuse them without refitting)
joblib.dump(imputer, "../artifacts/05_imputer.pkl")
joblib.dump(encoder, "../artifacts/05_encoder.pkl")

# Save the final feature list
feature_list = numeric_features + list(encoded_cols)
with open("../artifacts/05_feature_list.txt", "w") as f:
    f.write("\n".join(feature_list))

print("Saved all artifacts. Total features:", len(feature_list))

Saved all artifacts. Total features: 48


**Note:** Saved 3 feature tables (train/val/test), the fitted imputer 
and encoder (for reuse in production without refitting), and a feature 
list of 48 columns (9 numeric + 39 one-hot encoded) — intentionally 
excluding the target column is_late, which will be handled separately 
during training.